In [ ]:
# 1. Competition horizons 
times_eval = np.array([24.0, 48.0, 72.0])

# Ensure horizons are valid for validation fold
max_time = y_val["time"].max()
times_eval = times_eval[times_eval < max_time]

print("Using horizons:", times_eval)

# 2. Compute RSF + GBSA ensemble survival on validation
surv_rsf_val = rsf.predict_survival_function(X_val, return_array=True)
surv_gbsa_val = gbsa.predict_survival_function(X_val, return_array=True)

# Ensemble:  average
surv_val_ensemble = (surv_rsf_val + surv_gbsa_val) / 2

# 3. Extract survival probabilities at horizons 
time_indices = [np.searchsorted(rsf.unique_times_, t, side="right") - 1 for t in times_eval]
surv_at_times_val = np.vstack([surv_val_ensemble[:, idx] for idx in time_indices]).T

# 4. Compute Brier scores
_, brier_scores = brier_score(
    y_train,   
    y_val,
    surv_at_times_val,
    times_eval
)

# Print individual Brier scores
for t, score in zip(times_eval, brier_scores):
    print(f"Brier @{t}h:", score)

#  5. Compute C-index using ensemble risk scores
risk_scores_val = -surv_val_ensemble.mean(axis=1)
c_index = concordance_index_censored(y_val["event"], y_val["time"], risk_scores_val)[0]

#  6. Weighted Brier 
weights = {24.0: 0.3, 48.0: 0.4, 72.0: 0.3}
weighted_brier = float(np.sum([weights[t] * score for t, score in zip(times_eval, brier_scores)]))

# 7. Final hybrid score 
hybrid_score = 0.3 * c_index + 0.7 * (1 - weighted_brier)

print("C-index:", c_index)
print("Weighted Brier:", weighted_brier)
print("Hybrid Score:", hybrid_score)